In [1]:
import sys
import numpy as np

sys.path.append("../../../src/")
from Rain.Rain import Rain
sys.path.pop()

from keras.models import Sequential
from keras.layers import Dense, Activation, Dropout
import tensorflow as tf

c:\Users\Menna\AppData\Local\Programs\Python\Python38\lib\site-packages\scipy\__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.23.5
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [2]:
import os

def clean():
    folder_paths = ["logs", "../../../data/coord/", "../../../data/divider/", "../../../data/worker/"]  # Replace with the folder path you want to delete files from
    file_extensions = [".npy", ".pkl", ".log"]  # Replace with the file extension you want to delete
    for folder_path in folder_paths:
        if os.path.exists(folder_path):
            for filename in os.listdir(folder_path):
                for file_extension in file_extensions:
                    if filename.endswith(file_extension):
                        file_path = os.path.join(folder_path, filename)
                        os.remove(file_path)
clean()

In [3]:
config = {
  "mode": {
      "type": "local",
      "params": {
        "ips": ['127.0.0.1', '127.0.0.1', '127.0.0.1'],
        "ports": [50151, 50152, 50153]
      }
      
    },
  "temp_data_path": "../../../",
  "partitions": 3,
  "num_of_workers": 3,
  "iterations": 3,
  "learning_type": "DL",
  "DL": {
    "lib": {
      "type": "tensorflow",
      "params": {
        "loss": tf.keras.losses.CategoricalCrossentropy(),
        "optimizer": tf.keras.optimizers.Adam(learning_rate=0.001),

      }
    },
    "lr": 0.001,
    "epochs": 1,
    "batch_size": 128,
  }
}

In [4]:
def get_train_data():
    return np.load("../../../data/MNIST/train_data.npy"), np.load(
        "../../../data/MNIST/train_labels.npy"
    )


def get_test_data():
    return np.load("../../../data/MNIST/test_data.npy"), np.load(
        "../../../data/MNIST/test_labels.npy"
    )



In [5]:
def create_model():
    # network parameters
    hidden_units = 256
    dropout = 0.45
    input_size = 784
    num_labels = 10
    # model is a 3-layer MLP with ReLU and dropout after each layer
    model = Sequential()
    model.add(Dense(hidden_units, input_dim=input_size))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(hidden_units))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(num_labels))
    model.add(Activation("softmax"))
    return model

In [6]:
X_train, y_train = get_train_data()


In [7]:
model = create_model()
rain = Rain(config, model)

2023-07-05 00:27:41,572 [DEBUG] [Rain] Rain is initialized
2023-07-05 00:27:41,574 [DEBUG] [Provisioner] Creating coordinator
2023-07-05 00:27:41,576 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData\coord/
2023-07-05 00:27:41,578 [DEBUG] [Coordinator] Coordinator is initialized
2023-07-05 00:27:41,579 [DEBUG] [LocalProvisioner] Provisioner is initialized
2023-07-05 00:27:41,581 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData\divider/
2023-07-05 00:27:41,583 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData\divider/
2023-07-05 00:27:41,584 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData\divider/


In [8]:
model = rain.train(X_train, y_train, strategy='async')

2023-07-05 00:27:41,643 [DEBUG] [Rain] Creating workers
2023-07-05 00:27:41,669 [INFO] [Provisioner] provisioner is serving
2023-07-05 00:27:41,670 [DEBUG] [Provisioner] Starting coordinator
2023-07-05 00:27:41,674 [INFO] [Coordinator] coordinator is serving
2023-07-05 00:27:41,675 [DEBUG] [Coordinator] sending the num of workers to the provisioner
2023-07-05 00:27:41,684 [DEBUG] [Provisioner] Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
2023-07-05 00:27:41,692 [DEBUG] [Coordinator] sent Success receiving the number of workers to the provisioner
2023-07-05 00:27:41,695 [DEBUG] [LocalProvisioner] Creating 3 workers
2023-07-05 00:27:41,697 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData\worker/
2023-07-05 00:27:41,714 [INFO] [Worker_50151] Worker is running on port: 50151
2023-07-05 00:27:41,716 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData\worker/
2023-07-05 00:27:41,730 [INFO] [Worker_50

116/157 [=====================>........] - ETA: 0s - loss: 0.8086 - accuracy: 0.7442

2023-07-05 00:28:50,162 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker{worker_id}
2023-07-05 00:28:50,183 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData\divider/3_3_trained.pkl from worker3


157/157 [==============================] - 5s 13ms/step - loss: 0.7039 - accuracy: 0.7788
sending data to divider


2023-07-05 00:28:50,641 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker{worker_id}
2023-07-05 00:28:50,644 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData\divider/1_1_trained.pkl from worker1


...

2023-07-05 00:29:10,928 [DEBUG] [DeepLearning] Iteration 3/3 complete for worker 1.
DEBUG:DeepLearning:Iteration 3/3 complete for worker 1.


157/157 [==============================] - 3s 7ms/step - loss: 0.2547 - accuracy: 0.9254


2023-07-05 00:29:11,934 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker{worker_id}
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker{worker_id}
2023-07-05 00:29:11,936 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData\divider/2_2_trained.pkl from worker2
DEBUG:DividerAmbassador:divider begins downloading ../../..//RainData\divider/2_2_trained.pkl from worker2


sending data to divider


2023-07-05 00:29:12,337 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData\divider/2_2_trained.pkl from worker2 successfully
DEBUG:DividerAmbassador:Downloaded ../../..//RainData\divider/2_2_trained.pkl from worker2 successfully
2023-07-05 00:29:12,351 [DEBUG] [DeepLearning] Asynchronous update is done by worker 2
DEBUG:DeepLearning:Asynchronous update is done by worker 2
2023-07-05 00:29:12,417 [DEBUG] [DeepLearning] Iteration 3/3 complete for worker 2.
DEBUG:DeepLearning:Iteration 3/3 complete for worker 2.
2023-07-05 00:29:12,441 [DEBUG] [DividerAmbassador] divider ambassador stopped serving
DEBUG:DividerAmbassador:divider ambassador stopped serving
2023-07-05 00:29:12,444 [DEBUG] [Divider] Divider stopped serving
DEBUG:Divider:Divider stopped serving
2023-07-05 00:29:12,447 [DEBUG] [LocalProvisioner] Workers are deleted
DEBUG:LocalProvisioner:Workers are deleted
2023-07-05 00:29:12,448 [DEBUG] [Provisioner] [Deleted workers]
IPs : ['127.0.0.1', '127.0.0.1', '127.0.0.1'], por

In [11]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

79/79 [==============================] - 1s 3ms/step - loss: 0.1011 - accuracy: 0.9710

Test accuracy: 97.1%


In [10]:
model = rain.train(X_train, y_train, strategy='sync')

2023-07-05 00:29:13,222 [DEBUG] [Rain] Creating workers
DEBUG:Rain:Creating workers
2023-07-05 00:29:13,227 [INFO] [Provisioner] provisioner is serving
INFO:Provisioner:provisioner is serving
2023-07-05 00:29:13,229 [DEBUG] [Provisioner] Starting coordinator
DEBUG:Provisioner:Starting coordinator
2023-07-05 00:29:13,232 [ERROR] [Coordinator] Error in the coordinator server: Failed to bind to address [::]:50052; set GRPC_VERBOSITY=debug environment variable to see detailed error message.
ERROR:Coordinator:Error in the coordinator server: Failed to bind to address [::]:50052; set GRPC_VERBOSITY=debug environment variable to see detailed error message.
2023-07-05 00:29:13,254 [DEBUG] [LocalProvisioner] Creating 3 workers
DEBUG:LocalProvisioner:Creating 3 workers
2023-07-05 00:29:13,257 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData\worker/
DEBUG:TemporaryFilesManager:Created temporary directory ../../..//RainData\worker/
2023-07-05 00:29:13,262 [ERROR] [Wor

116/157 [=====================>........] - ETA: 0s - loss: 0.2125 - accuracy: 0.9390

2023-07-05 00:30:49,915 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker{worker_id}


sending data to divider


DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker{worker_id}
2023-07-05 00:30:49,919 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData\divider/3_1_trained.pkl from worker3
DEBUG:DividerAmbassador:divider begins downloading ../../..//RainData\divider/3_1_trained.pkl from worker3


...

157/157 [==============================] - 6s 17ms/step - loss: 0.1671 - accuracy: 0.9492


2023-07-05 00:31:07,882 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker{worker_id}


sending data to divider


DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker{worker_id}
2023-07-05 00:31:07,889 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData\divider/2_3_trained.pkl from worker2
DEBUG:DividerAmbassador:divider begins downloading ../../..//RainData\divider/2_3_trained.pkl from worker2
2023-07-05 00:31:08,420 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData\divider/3_3_trained.pkl from worker3 successfully
DEBUG:DividerAmbassador:Downloaded ../../..//RainData\divider/3_3_trained.pkl from worker3 successfully
2023-07-05 00:31:08,718 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData\divider/1_3_trained.pkl from worker1 successfully
DEBUG:DividerAmbassador:Downloaded ../../..//RainData\divider/1_3_trained.pkl from worker1 successfully
2023-07-05 00:31:08,766 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData\divider/2_3_trained.pkl from worker2 successfully
DEBUG:DividerAmbassador:Downloaded ../../..//RainData\d

AttributeError: workers

In [ ]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

79/79 [==============================] - 1s 4ms/step - loss: 0.0996 - accuracy: 0.9690

Test accuracy: 96.9%
